# 2 — A rede de regiões de Campinas

**A unidade de análise é a antena.** Cada antena é uma região da cidade; as pessoas que
moram sob ela entram como atributos agregados, e as chamadas entre dois moradores da mesma
antena deixam de ser arestas e viram a *insularidade* daquela região.

Este notebook usa os mesmos módulos do pipeline (`src/`), então os números aqui e os de
`output/<cidade>/summary/report.md` são sempre os mesmos.

## Preparação

In [ ]:
import sys
from pathlib import Path

# permite rodar o notebook a partir de notebooks/ usando os módulos de src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils import load_config
from src.exporter import InlineExporter
from src import antenna

CIDADE = "campinas"   # troque aqui: precisa existir config/<cidade>.yaml

config = load_config(CIDADE)
config["spatial"]["download_basemap"] = True   # False para rodar offline

edges_antenna = pd.read_parquet(ROOT / config["data"]["edges_antenna_path"])
antennas = pd.read_parquet(ROOT / config["data"]["antennas_path"])

net = antenna.build(edges_antenna, antennas, config)
ex = InlineExporter(config)
print(f"{net.n_antennas} regiões | {net.G.number_of_edges()} fluxos | "
      f"{net.nodes['n_users'].sum():,} moradores agregados")

## A tabela de nós

Cada linha é uma região. Repare em `calls_internal` e `insularity`: é para onde foram as
chamadas que antes eram arestas dentro da mesma antena — mais de um terço do volume da cidade.

In [ ]:
net.nodes[[
    "antenna_id", "n_users", "calls_total", "calls_internal", "insularity",
    "net_balance", "calls_per_user", "residence_quintile_state",
]].sort_values("calls_total", ascending=False).head(10)

In [ ]:
print(f"chamadas que não saem da região de origem: {net.internal_call_share:.1%}")
net.nodes[["n_users", "calls_total", "insularity", "net_balance"]].describe().round(3)

## A tabela de fluxos

`n_pairs` distingue um corredor *largo* (muita gente conversando pouco) de um *estreito*
(poucos laços intensos). `intensity` normaliza pelo tamanho das duas regiões — sem isso, o
mapa de fluxos apenas redesenha onde mora mais gente.

In [ ]:
net.flows[["a", "b", "q_calls", "n_pairs", "dist_km", "intensity"]].head(10)

## Por que a caixa de ferramentas muda

A rede tem ~145 nós e densidade ≈ 0,56: **mais da metade dos pares de regiões da cidade tem
contato**. Com isso, grau deixa de distinguir qualquer coisa — e junto com ele caem lei de
potência, small-world contra Erdős–Rényi, k-core e componente gigante, que eram o núcleo da
análise no nível de usuário. O que informa agora é **peso**: força, desigualdade de volume e
o *backbone* dos fluxos estatisticamente significativos.

In [ ]:
import networkx as nx

print(f"densidade: {nx.density(net.G):.3f}")
graus = pd.Series(dict(net.G.degree()))
print(f"grau: mediana {graus.median():.0f}, mínimo {graus.min()}, máximo {graus.max()}")
print(f"caminho médio: {nx.average_shortest_path_length(net.G):.2f}  (quase todo mundo é vizinho)")

## Estrutura: força, backbone, macro-regiões, s-core e balanço

A célula abaixo roda o módulo de topologia do pipeline e mostra tudo inline.

In [ ]:
from src.pipeline import topology

resultado = topology.run(net, config, ex)
nodes = resultado["nodes"]   # inclui macro_region e score_level

### Os corredores mais fortes da cidade

O backbone é o esqueleto: ~11% dos fluxos que carregam ~62% de todas as chamadas.

In [ ]:
ex.data["backbone_flows.csv"].head(15)

### As macro-regiões funcionais

Louvain ponderado sobre os fluxos divide a cidade em poucas macro-regiões. O teste
interessante vem no notebook 3: elas saem **espacialmente contíguas** no mapa, mesmo o
algoritmo não sabendo nada sobre geografia.

In [ ]:
nodes.groupby("macro_region").agg(
    regioes=("antenna_id", "size"),
    moradores=("n_users", "sum"),
    chamadas=("calls_total", "sum"),
    insularidade_media=("insularity", "mean"),
).round(3)